# 🔧 Gearbox Health — Explainability & Maintenance Cost Analysis
**DSMLC Final Competition 2026 | Enbridge Wind Turbine SCADA Analysis**

### Research Question 1c
> *How might explainable models reduce unnecessary maintenance while catching issues months ahead?*

### Sections
1. Setup & Data Loading
2. Why Our Model is Explainable
3. Maintenance Cost Analysis
4. Explainable Alert Dashboard
5. Summary & Recommendations

---
## Section 1 — Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

plt.rcParams['figure.figsize'] = (15, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print('✅ Imports OK')

In [ ]:
# ⚠️ UPDATE THIS PATH
BASE_PATH = Path('/Users/prabhaseessingh/Desktop/Data Science/DSMLCFinalComp/CARE_To_Compare/Wind Farm A')

DATASETS_PATH     = BASE_PATH / 'datasets'
EVENT_INFO_PATH   = BASE_PATH / 'event_info.csv'
FEATURE_DESC_PATH = BASE_PATH / 'feature_description.csv'

event_info = pd.read_csv(EVENT_INFO_PATH, sep=';')
feat_desc  = pd.read_csv(FEATURE_DESC_PATH, sep=';')

def get_label(bare_name):
    d = feat_desc[feat_desc['sensor_name'] == bare_name]['description'].values
    return d[0] if len(d) > 0 else bare_name

# Analysis parameters — same as gearbox_analysis notebook
DRIFT_WINDOW   = 1008   # 7-day rolling window
BASELINE_STEPS = 4320   # first 30 days
MULTIPLIER     = 2.3
SENSOR_COL     = 'sensor_12_avg'
SENSOR_BARE    = 'sensor_12'
TARGET_EVENT   = 72

# Load primary gearbox event
df_anomaly = pd.read_csv(DATASETS_PATH / f'{TARGET_EVENT}.csv', sep=';')
df_anomaly = df_anomaly.sort_values('id').reset_index(drop=True)
df_train   = df_anomaly[df_anomaly['train_test'] == 'train'].copy()
TRAIN_END  = df_train.index.max()

# Compute rolling mean and threshold
rolling      = df_anomaly[SENSOR_COL].rolling(
    window=DRIFT_WINDOW, min_periods=DRIFT_WINDOW//2
).mean()
baseline     = rolling.iloc[:BASELINE_STEPS].dropna()
BASE_MEAN    = baseline.mean()
BASE_STD     = baseline.std()
THRESHOLD    = BASE_MEAN + MULTIPLIER * BASE_STD

# Find first alert
pre_fault    = rolling[rolling.index < TRAIN_END]
exceeded     = pre_fault[pre_fault > THRESHOLD]
FIRST_ALERT  = exceeded.index[0] if len(exceeded) > 0 else None
DAYS_EARLY   = (TRAIN_END - FIRST_ALERT) * 10 / 60 / 24 if FIRST_ALERT else None

print(f'Event {TARGET_EVENT} loaded: {len(df_anomaly):,} rows')
print(f'Sensor: {get_label(SENSOR_BARE)}')
print(f'Baseline mean: {BASE_MEAN:.1f}°C | Threshold: {THRESHOLD:.1f}°C')
print(f'First alert: {DAYS_EARLY:.1f} days before fault' if DAYS_EARLY else 'No alert detected')

---
## Section 2 — Why Our Model is Explainable
Unlike black-box models (neural networks, random forests), our approach
gives a clear, human-readable reason for every single alert it produces.
This section demonstrates exactly what that explanation looks like.

In [ ]:
# Simulate what happens at the moment of alert — what can we tell the engineer?
if FIRST_ALERT:
    current_rolling_val = rolling.loc[FIRST_ALERT]
    deviation_c         = current_rolling_val - BASE_MEAN
    deviation_sigma     = deviation_c / BASE_STD
    days_of_drift       = 7  # by definition — our window is 7 days

    print('=' * 62)
    print('  WHAT THE ENGINEER SEES WHEN THE ALERT FIRES')
    print('=' * 62)
    print(f'  Turbine:          21 (Event {TARGET_EVENT})')
    print(f'  Sensor:           {get_label(SENSOR_BARE)}')
    print(f'  Alert type:       Sustained temperature drift')
    print(f'  Baseline avg:     {BASE_MEAN:.1f}°C (from first 30 days)')
    print(f'  Current 7-day avg:{current_rolling_val:.1f}°C')
    print(f'  Deviation:        +{deviation_c:.1f}°C above baseline')
    print(f'                    ({deviation_sigma:.1f} standard deviations)')
    print(f'  Alert threshold:  {THRESHOLD:.1f}°C (baseline + {MULTIPLIER}σ)')
    print(f'  Duration of drift:{days_of_drift} days sustained above normal')
    print(f'  Time to fault:    ~{DAYS_EARLY:.0f} days (estimated)')
    print(f'  Recommended:      Schedule gearbox inspection within 30 days')
    print('=' * 62)
    print()
    print('  WHY THIS IS EXPLAINABLE:')
    print('  Every number above can be traced back to raw sensor data.')
    print('  No hidden weights. No black-box layers.')
    print('  An engineer can verify this finding in 30 seconds.')

In [ ]:
# Compare explainability: our method vs a hypothetical black-box model
comparison = {
    'Property': [
        'Alert reason visible?',
        'Engineer can verify?',
        'Requires ML expertise?',
        'Adapts to each turbine?',
        'Catches drift early?',
        'Can tune false alarm rate?',
        'Regulatory audit trail?'
    ],
    'Our Method (Rolling Mean + Threshold)': [
        '✅ Yes — exact sensor & deviation shown',
        '✅ Yes — raw data traceable',
        '✅ No — intuitive statistics',
        '✅ Yes — per-turbine baseline',
        '✅ Yes — 53 days early',
        '✅ Yes — adjust multiplier',
        '✅ Yes — full transparency'
    ],
    'Black-Box Model (e.g. Neural Network)': [
        '❌ No — confidence score only',
        '❌ No — weights not interpretable',
        '❌ Yes — needs ML team',
        '⚠️  Requires retraining',
        '⚠️  Depends on training data',
        '⚠️  Requires threshold tuning',
        '❌ Difficult to audit'
    ]
}

comp_df = pd.DataFrame(comparison)
print('Explainability Comparison:')
comp_df

---
## Section 3 — Maintenance Cost Analysis
Quantifying the financial value of catching faults early vs. reacting after failure,
and the cost of false alarms from an overly sensitive threshold.

In [ ]:
# Industry cost assumptions for onshore wind turbines
# Sources: NREL wind O&M cost reports, general industry estimates
COST_UNPLANNED_FAILURE   = 150_000  # $ — emergency repair + lost revenue + crane hire
COST_PLANNED_MAINTENANCE =  15_000  # $ — scheduled gearbox inspection + repair
COST_FALSE_ALARM_CALLOUT =   5_000  # $ — unnecessary technician dispatch

timesteps_total   = len(df_train)
timesteps_per_day = 144

scenarios = [
    {'name': 'No monitoring\n(unplanned failure)',
     'fp_rate': 0.0, 'detected': False,
     'maintenance_cost': COST_UNPLANNED_FAILURE},
    {'name': 'Noisy threshold\n(1.5σ)',
     'fp_rate': 0.30, 'detected': True,
     'maintenance_cost': COST_PLANNED_MAINTENANCE},
    {'name': 'Our method\n(2.3σ)',
     'fp_rate': 0.01, 'detected': True,
     'maintenance_cost': COST_PLANNED_MAINTENANCE},
]

print('=' * 65)
print('MAINTENANCE COST COMPARISON ACROSS MONITORING STRATEGIES')
print('=' * 65)

total_costs = []
for s in scenarios:
    fp_steps    = timesteps_total * s['fp_rate']
    fp_days     = fp_steps / timesteps_per_day
    callouts    = fp_days / 3  # assume callout every 3 days of sustained flagging
    callout_cost = callouts * COST_FALSE_ALARM_CALLOUT
    total        = callout_cost + s['maintenance_cost']
    total_costs.append(total)

    print(f"\n📋 {s['name'].replace(chr(10), ' ')}")
    print(f'   False alarm rate:    {s["fp_rate"]*100:.0f}%')
    print(f'   Unnecessary callouts: ~{callouts:.0f}')
    print(f'   Callout cost:        ${callout_cost:>10,.0f}')
    print(f'   Maintenance cost:    ${s["maintenance_cost"]:>10,.0f}')
    print(f'   TOTAL:               ${total:>10,.0f}')

print(f'\n💰 Savings (our method vs unplanned): ${total_costs[0] - total_costs[2]:,.0f}')
print(f'💰 Savings (our method vs noisy):     ${total_costs[1] - total_costs[2]:,.0f}')
print('=' * 65)

In [ ]:
# Visualise the cost comparison
labels   = ['No monitoring\n(unplanned failure)', 'Noisy threshold\n(1.5σ)', 'Our method\n(2.3σ)']
colors   = ['tomato', 'darkorange', 'seagreen']

# Break down costs into stacked components
maintenance_costs = [COST_UNPLANNED_FAILURE, COST_PLANNED_MAINTENANCE, COST_PLANNED_MAINTENANCE]
callout_costs = []
for s in scenarios:
    fp_steps = timesteps_total * s['fp_rate']
    callouts = (fp_steps / timesteps_per_day) / 3
    callout_costs.append(callouts * COST_FALSE_ALARM_CALLOUT)

fig, ax = plt.subplots(figsize=(10, 6))

x = range(len(labels))
bars1 = ax.bar(x, maintenance_costs, color=colors, alpha=0.85, label='Maintenance/failure cost')
bars2 = ax.bar(x, callout_costs, bottom=maintenance_costs,
               color=colors, alpha=0.4, hatch='//', label='False alarm callout cost')

# Add total labels on top
for i, (m, c) in enumerate(zip(maintenance_costs, callout_costs)):
    total = m + c
    ax.text(i, total + 2000, f'${total:,.0f}',
            ha='center', va='bottom', fontweight='bold', fontsize=10)

# Savings arrow
ax.annotate('', xy=(2, total_costs[2]), xytext=(0, total_costs[0]),
            arrowprops=dict(arrowstyle='<->', color='black', lw=1.5))
mid_y = (total_costs[0] + total_costs[2]) / 2
ax.text(1, mid_y, f'Save ${total_costs[0]-total_costs[2]:,.0f}',
        ha='center', va='center', fontsize=10, fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', edgecolor='black'))

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel('Estimated Cost (USD)')
ax.set_title('Maintenance Cost Comparison: Monitoring Strategies\n'
             'Explainable early detection significantly reduces total cost',
             fontweight='bold')
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.savefig('maintenance_cost_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: maintenance_cost_comparison.png')

---
## Section 4 — Explainable Alert Dashboard
A zoomed-in view of the 90 days before the fault — showing exactly what
a maintenance engineer would see on their monitoring dashboard when an alert fires.

In [ ]:
# Zoom into 90 days before fault for a clean, readable chart
ZOOM_DAYS_BEFORE = 90
ZOOM_DAYS_AFTER  = 7
zoom_start = max(0, TRAIN_END - ZOOM_DAYS_BEFORE * timesteps_per_day)
zoom_end   = min(len(df_anomaly) - 1, TRAIN_END + ZOOM_DAYS_AFTER * timesteps_per_day)

df_zoom      = df_anomaly.loc[zoom_start:zoom_end]
rolling_zoom = rolling.loc[zoom_start:zoom_end]

fig, ax = plt.subplots(figsize=(15, 7))

# Raw signal (faded)
ax.plot(df_zoom.index, df_zoom[SENSOR_COL],
        color='steelblue', linewidth=0.5, alpha=0.25, label='Raw sensor reading')

# 7-day rolling mean
ax.plot(df_zoom.index, rolling_zoom,
        color='steelblue', linewidth=2.5, label='7-day rolling mean')

# Threshold
ax.axhline(y=THRESHOLD, color='red', linestyle='--', linewidth=1.5,
           label=f'Alert threshold ({THRESHOLD:.1f}°C)')

# Baseline mean
ax.axhline(y=BASE_MEAN, color='steelblue', linestyle=':', linewidth=1.2,
           label=f'Turbine baseline ({BASE_MEAN:.1f}°C)')

# Alert and fault markers
if FIRST_ALERT and FIRST_ALERT >= zoom_start:
    ax.axvline(x=FIRST_ALERT, color='red', linewidth=2,
               label=f'⚠️ Alert triggered ({DAYS_EARLY:.0f} days before fault)')
    ax.axvspan(FIRST_ALERT, TRAIN_END, alpha=0.08, color='red', label='Inspection window')

ax.axvline(x=TRAIN_END, color='black', linestyle='--',
           linewidth=1.8, label='Fault occurs')

# Engineer-readable annotation box
if FIRST_ALERT and FIRST_ALERT >= zoom_start:
    current_val = rolling.loc[FIRST_ALERT]
    annotation_x = FIRST_ALERT - 2000 if FIRST_ALERT - 2000 >= zoom_start else FIRST_ALERT + 500
    ax.annotate(
        f'⚠️  MAINTENANCE ALERT\n'
        f'Sensor: Gearbox Oil Temperature\n'
        f'7-day avg: {current_val:.1f}°C\n'
        f'Baseline: {BASE_MEAN:.1f}°C\n'
        f'Deviation: +{current_val - BASE_MEAN:.1f}°C ({MULTIPLIER}σ)\n'
        f'Est. time to fault: ~{DAYS_EARLY:.0f} days\n'
        f'Action: Schedule inspection',
        xy=(FIRST_ALERT, current_val),
        xytext=(annotation_x, current_val + 5),
        fontsize=9, family='monospace',
        bbox=dict(boxstyle='round,pad=0.6', facecolor='lightyellow',
                  edgecolor='red', linewidth=1.5),
        arrowprops=dict(arrowstyle='->', color='red', lw=1.5)
    )

ax.set_title(
    'Explainable Alert Dashboard — 90-Day Window Before Gearbox Fault\n'
    'Turbine 21 | Event 72 | Gearbox Oil Temperature (sensor_12)',
    fontweight='bold', fontsize=11
)
ax.set_xlabel('Time step (10-min intervals)')
ax.set_ylabel('Temperature (°C)')
ax.legend(fontsize=9, loc='upper left')
plt.tight_layout()
plt.savefig('explainable_alert_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: explainable_alert_dashboard.png')

---
## Section 5 — Summary & Recommendations

In [ ]:
print('=' * 65)
print('EXPLAINABILITY & COST ANALYSIS — KEY FINDINGS (Question 1c)')
print('=' * 65)

print('\n🔍 Why Our Model is Explainable:')
print('   Every alert includes: sensor name, current value,')
print('   baseline reference, deviation in °C and sigma,')
print('   and a recommended action. No black-box outputs.')

print('\n⏱️  Early Detection Performance:')
print(f'   Fault detected {DAYS_EARLY:.0f} days before failure (Turbine 21, Event 72)')
print(f'   Threshold: {THRESHOLD:.1f}°C (baseline {BASE_MEAN:.1f}°C + {MULTIPLIER}σ)')
print(f'   False positive rate at 2.3σ: ~1%')

print('\n💰 Financial Impact (per turbine per fault):')
print(f'   Unplanned failure cost:    ${COST_UNPLANNED_FAILURE:>10,.0f}')
print(f'   Our method total cost:     ${total_costs[2]:>10,.0f}')
print(f'   Estimated savings:         ${COST_UNPLANNED_FAILURE - total_costs[2]:>10,.0f}')

print('\n✅ Operational Recommendations:')
print('   1. Deploy rolling mean monitor on sensor_12 for all turbines')
print('   2. Use per-turbine baselines — never a fixed fleet-wide threshold')
print('   3. Set multiplier at 2.3σ to balance earliness vs false alarms')
print('   4. Show engineers the full alert explanation — not just a flag')
print('   5. Cross-validate alerts with sensor_11 before dispatching crew')
print('   6. Re-baseline after any major maintenance or turbine replacement')
print('=' * 65)